# Cryptography (CC4017) -- Week 10


## Q1: Man-in-the-Middle

Implement a prototype that demonstrates how a Man-in-the-Middle attack can occur in a standard
unauthenticated Diffie-Hellman key exchange. As usual, this happens between usual suspects Alice
and Bob. In appendix, you can find four files:

- config_alice and config_bob are configuration files that are used by Alice and Bob to know
who to talk to.
- alice.py establishes a connection with Bob (hopefully), and performs the Diffie-Hellman key
exchange.
- bob.py mirrors the behavior of Alice.

The goal of the work is to design the man-in-the-middle adversary code: mitm.py. Your code must
convince Alice and Bob to instead talk to him, and perform a key exchange with him. Your attack
must not change the source code of Alice or Bob. Your attack is successful if Alice and Bob
are not agreeing on the same secret, and the secrets they have agreed to are both known to the
adversary.

Suggestion: Start by analysing the code for Alice and Bob, what are they using to communicate?
How can we subvert this mechanism to be more. . . convenient?
To facilitate communication, this code uses pwntools (reference). It is not mandatory to use this, but
the library considerably facilitates communication.

The program mitm.py handles two threads of communication, one with alice where the eavesdropper mimics bob and one with bob where alice is mimicked. By sending alice and bob its own exponent it makes it so it successfully establishes a connection between the both of them acting as middle man, where bob and alice do not agree on the shared secret but can communicate through the eavesdropper, making it a successful MITM attack

To test it execute alice.py first, then mitm.py and then bob.py


## Q2: ECC
The following is a naive attempt at an elliptic curve signature scheme. Consider a global elliptic curve,
prime p and generator G. The scheme works as follows.
- Alice picks a private signing key skA and forms the public verifying key by computing pkA ← skA ·G
- To sign message m, Alice picks a random value k, and computes the signature σ ← m − k · skA · G.
It then sends to Bob the tuple (m, k, σ)
- To verify the signature, Bob checks that m = σ + k · pkA . If this is true, the signature is validated.

### P1:
Show that the scheme works, i.e. show that, for correctly signed messages, the
verification algorithm works accordingly.

In order to validate that this scheme works we have to guarantee that the m coming out of the verification is the same as the m that is being signed:

m = σ + k * pkA, since σ = m - k * skA * G, 

m = m - k * skA * G + * k * pkA, with pkA = skA * G,

m = m - k * pkA + k * pkA <=> m = m

proving that the verifcation algorithm works correctly


### P2:
Show that this scheme is vulnerable, by describing a simple technique for forging
a signature on an arbitrary message, without knowledge of the secret key skA . Hint: consider what
computations can one do using simply pkA


According to the scheme, σ = m - k * skA * G, since we know that skA * G = pkA, this implies that anyone who knows the public key, is capable of not only validating signatures but also signing their own messages.

## Q3: ElGamal
ElGamal is a public-key encryption scheme. Its reasoning is similar to that of classical Diffie-Hellman,
using g x as the public key, and having the encryption encapsulate the message with g y .
The algorithms are presented below, assuming operations in the group Zq , with generator g. Encryption
assumes that m is an element of Zq , whcih can be achieved by having a reversible mapping function
from the message domain to the group domain. s−1 means the inverse of s.

### P1: 
Describe why decryption works – show how m is recovered

To prove the decryption works, we need to guarantee that the initial encryption messaged is going to be equal,mathematically, to the final message after decryption. In this scheme we have:

m = c * s2⁻¹ -> for decryption. With this s2 being Y^x to avoid confusion. As such,

m = c * (Y^x)⁻¹, with c being m * s1,

m = m * s1 * (Y^x)⁻¹, and since s1 is X^y,

m = m * X^y * (Y^x)⁻¹. Since X is g^x and Y is g^y, we obtain:

m = m * g^(xy) * (g^(yx))⁻¹. As long as g is prime with the modulus being used in the calculations,

m = m * 1 , leading to, m = m

### P2:
Explain how the hardness of the discrete logarithm ensures confidentiality

The dlp ensures the confidentiality of the cryptographic scheme by making it computanionally infeasible for an attacker or eavesdropper to derive what the secret key is from public information / the public key. If the numbers used are properly picked, they require exponetial time or sub-exponential time to compute, making it impossible for larger key sizes. An attacker that observes g, X or Y will have an hard time computing the values of x and y, making it impossible to find out the share secret through the method of finding the secret exponents. 

### P3:
ElGamal is malleable – show how it can be done. Consider an adversary that can
request the encryption of message m, receiving ciphertext c, and show how can he present another
ciphertext c′ that will decrypt in a related way.

An attacker can very easily present a c' that's related to c the same way m',its decryption, will be related to m. Due to how the scheme  works, an attacker that requests m to be encrypted and obtains c, can then multiply c by any number n, and the m obtained by the decryption will be equal to n * m. This is because of how the scheme decrypts messages. When presented with c' = c * n:

m = c' * s2⁻¹, which leads to m = n * c * s2⁻¹ <=> m = n * m * g^(xy) * (g^(yx))⁻¹, meaning that the new m = n * m, just as c' = n * c